# Una convolución por dentro

**Explorador de Hespérides · Capítulo 4**

Ampliación programada sobre los conceptos de los notebooks D2L de este capítulo.

La ventana recorre 36 posiciones. Observa los productos locales y su suma: cada uno corresponde exactamente a una celda del mapa de respuesta. La operación utiliza stride 1 y padding 0. Cambia el filtro para distinguir respuesta a orientación de simple suavizado.

![Ilustración conceptual](../recursos/ilustraciones/capitulo_4.png)

*Ilustración conceptual generada con ImageGen. Los resultados cuantitativos son los del código.*

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# Cada posición usa los mismos nueve pesos. PyTorch calcula correlación cruzada.
imagen = np.zeros((8,8));imagen[1:7,3:6] = 1
filtros = {'Borde vertical': np.array([[-1,0,1],[-1,0,1],[-1,0,1]]),
           'Borde horizontal': np.array([[-1,-1,-1],[0,0,0],[1,1,1]]),
           'Promedio': np.ones((3,3))/9}

def ver_convolucion(posicion=14, filtro='Borde vertical'):
    K = filtros[filtro]
    salida = nn.functional.conv2d(torch.tensor(imagen)[None,None],torch.tensor(K,dtype=torch.float64)[None,None])[0,0].numpy()
    fila,col = divmod(posicion,6)
    parche = imagen[fila:fila+3,col:col+3]
    producto = parche*K
    assert np.isclose(producto.sum(),salida[fila,col])
    fig, axes = plt.subplots(1,4,figsize=(13,3))
    matrices=[imagen,K,producto,salida]
    titulos=['Entrada y ventana','Filtro compartido','Productos locales',f'Suma = {producto.sum():.2f}']
    for ax,M,titulo in zip(axes,matrices,titulos):
        ax.imshow(M,cmap='coolwarm',vmin=-3,vmax=3)
        ax.set_title(titulo);ax.set_xticks([]);ax.set_yticks([])
        if M.shape==(3,3):
            for (i,j),v in np.ndenumerate(M):ax.text(j,i,f'{v:.2g}',ha='center',va='center')
    axes[0].add_patch(plt.Rectangle((col-.5,fila-.5),3,3,fill=False,edgecolor='#D99B18',lw=3))
    axes[3].add_patch(plt.Rectangle((col-.5,fila-.5),1,1,fill=False,edgecolor='#D99B18',lw=3))
    fig.tight_layout();plt.show()

interact(ver_convolucion,posicion=widgets.IntSlider(value=14,min=0,max=35,description='Ventana',continuous_update=False),
         filtro=list(filtros));


## Vista de referencia

Esta figura conserva el estado inicial también en una exportación sin kernel. Los controles anteriores se utilizan en Jupyter.

In [ ]:
ver_convolucion(14, 'Borde vertical')

## El proceso en movimiento

Puedes reproducir, pausar y recorrer los fotogramas. La animación se genera a partir de los estados calculados arriba.

In [ ]:

K=filtros['Borde vertical']
salida=nn.functional.conv2d(torch.tensor(imagen)[None,None],torch.tensor(K,dtype=torch.float64)[None,None])[0,0].numpy()
fig,axes=plt.subplots(1,2,figsize=(7,3.5))
axes[0].imshow(imagen,cmap='gray',vmin=0,vmax=1)
ventana=plt.Rectangle((-.5,-.5),3,3,fill=False,edgecolor='#D99B18',lw=3);axes[0].add_patch(ventana)
mapa=axes[1].imshow(np.full((6,6),np.nan),cmap='coolwarm',vmin=-3,vmax=3)
axes[0].set_title('El mismo filtro en cada posición');axes[1].set_title('Mapa de respuesta')
def avanzar_ventana(paso):
    fila,col=divmod(paso,6);ventana.set_xy((col-.5,fila-.5))
    parcial=np.full((6,6),np.nan);parcial.flat[:paso+1]=salida.flat[:paso+1];mapa.set_data(parcial)
    return ventana,mapa
animacion=FuncAnimation(fig,avanzar_ventana,frames=36,interval=180)
plt.close(fig);display(HTML(animacion.to_jshtml()))


## Comprobación

Modifica un control cada vez y describe qué cambia y qué permanece constante. Compara tu observación con las preguntas del capítulo.